In [5]:
"""
WBPR-LightGCN Retrain + 评估脚本
================================================
流程：
  1. 从 influence_matrix + tau_u 推导每位用户的 S_u
  2. 构建 D_remove = {(u,i) | i ∈ B_u ∩ I_k, k ∈ S_u}
  3. 在 D_train \ D_remove 上重跑 WBPR-LightGCN
  4. 保存重训练 embedding
  5. 计算评估指标（对比 Original / Unlearning / Retrain）
================================================
"""

import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import scipy.sparse as sp
import pandas as pd
from collections import defaultdict

# ─────────────────────────────────────────────────────────────
# 路径
# ─────────────────────────────────────────────────────────────
DATA_PATH      = "/Users/yubinhao/ml-1m/Amazon Review/数据集/Amazon_Reviews_Processed"
EMBED_PATH     = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/模型文件/embeddings_best.pt"
MAPPING_PATH   = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/模型文件/id_mappings.pt"
CONFIG_PATH    = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/模型文件/config.pt"
ITEM_SETS_PATH = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_basis/pattern_item_sets.pt"
INFLUENCE_PATH = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_influence/influence_scores.pt"
UNLEARN_PATH   = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/遗忘实验/embeddings_unlearned.pt"
THRESHOLD_PATH = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/遗忘实验/threshold_info.pt"
SAVE_DIR       = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/Retrain"

# 与原始训练完全一致的超参数
EMBED_DIM        = 64
N_LAYERS         = 3
LR               = 1e-3
WEIGHT_DECAY     = 1e-4
BATCH_SIZE       = 2048
EPOCHS           = 100
LOG_EVERY        = 10
RATING_MAX       = 5.0
MIN_INTERACTIONS = 5
SEED             = 42
TOP_K            = 8
EPS              = 1e-10

os.makedirs(SAVE_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────
# 工具
# ─────────────────────────────────────────────────────────────
def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


# ─────────────────────────────────────────────────────────────
# 1. 加载已有数据
# ─────────────────────────────────────────────────────────────
print("[1] 加载数据...")
set_seed(SEED)
device = get_device()
print(f"  设备: {device}")

mapping        = torch.load(MAPPING_PATH,   map_location="cpu", weights_only=False)
user2idx       = mapping["user2idx"]
item2idx       = mapping["item2idx"]
idx2user       = mapping["idx2user"]
idx2item       = mapping["idx2item"]
n_users        = len(user2idx)
n_items        = len(item2idx)

sets_ckpt         = torch.load(ITEM_SETS_PATH, map_location="cpu", weights_only=False)
pattern_item_sets = sets_ckpt["pattern_item_sets"]

infl_ckpt        = torch.load(INFLUENCE_PATH, map_location="cpu", weights_only=False)
influence_matrix = infl_ckpt["influence_matrix"]
sampled_uids     = infl_ckpt["sampled_uids"]
hq_patterns      = infl_ckpt["hq_patterns"]

thresh_ckpt = torch.load(THRESHOLD_PATH, map_location="cpu", weights_only=False)
tau_u       = thresh_ckpt["tau_u"]

unlearn_ckpt     = torch.load(UNLEARN_PATH, map_location="cpu", weights_only=False)
user_emb_orig    = unlearn_ckpt["user_emb_before"]
user_emb_unlearn = unlearn_ckpt["user_emb_after"]

orig_ckpt     = torch.load(EMBED_PATH, map_location="cpu", weights_only=False)
item_emb_orig = orig_ckpt["item_embeddings"].numpy()

print(f"  n_users={n_users}, n_items={n_items}")


# ─────────────────────────────────────────────────────────────
# 2. 构建原始训练数据
# ─────────────────────────────────────────────────────────────
print("\n[2] 读取原始训练数据...")
df = pd.read_csv(DATA_PATH, usecols=["user_id", "item_id", "rating"])
user_counts = df.groupby("user_id").size()
valid_users = user_counts[user_counts >= MIN_INTERACTIONS].index
df = df[df["user_id"].isin(valid_users)].copy()
df["uid"] = df["user_id"].map(user2idx)
df["iid"] = df["item_id"].map(item2idx)
print(f"  原始训练集大小: {len(df)}")

# B_u
user_item_sets = df.groupby("uid")["iid"].apply(set).to_dict()


# ─────────────────────────────────────────────────────────────
# 3. 构建 S_u 与 D_remove
# ─────────────────────────────────────────────────────────────
print("\n[3] 构建 S_u 与 D_remove...")

S_u_dict = {}
for idx, uid in enumerate(sampled_uids):
    scores    = influence_matrix[idx]
    sorted_ki = np.argsort(scores)[::-1]
    targets   = [hq_patterns[ki] for ki in sorted_ki if scores[ki] > tau_u[idx]]
    S_u_dict[uid] = targets

# D_remove：(uid, iid) 对的集合
D_remove = set()
for uid, patterns in S_u_dict.items():
    if len(patterns) == 0:
        continue
    B_u = user_item_sets.get(uid, set())
    for k in patterns:
        I_k = set(pattern_item_sets[k])
        for iid in B_u & I_k:
            D_remove.add((uid, iid))

print(f"  D_remove 大小: {len(D_remove)} 条交互")


# ─────────────────────────────────────────────────────────────
# 4. 构建 D_retrain = D_train \ D_remove
# ─────────────────────────────────────────────────────────────
print("\n[4] 构建 D_retrain...")
mask = ~df.apply(lambda row: (row["uid"], row["iid"]) in D_remove, axis=1)
df_retrain = df[mask].copy()
print(f"  D_retrain 大小: {len(df_retrain)} 条（删除了 {len(df) - len(df_retrain)} 条）")


# ─────────────────────────────────────────────────────────────
# WBPR-LightGCN 模型定义
# ─────────────────────────────────────────────────────────────
def build_normalized_adjacency(df, n_users, n_items, rating_max=5.0):
    uids    = df["uid"].values
    iids    = df["iid"].values
    weights = (df["rating"].values / rating_max).astype(np.float32)
    N       = n_users + n_items
    all_row = np.concatenate([uids,           iids + n_users])
    all_col = np.concatenate([iids + n_users, uids          ])
    all_w   = np.concatenate([weights,        weights       ])
    A = sp.csr_matrix((all_w, (all_row, all_col)), shape=(N, N), dtype=np.float32)
    d_w          = np.array(A.sum(axis=1)).flatten()
    d_w_inv_sqrt = np.where(d_w > 0, 1.0 / np.sqrt(d_w), 0.0)
    D_inv_sqrt   = sp.diags(d_w_inv_sqrt)
    hat_A        = D_inv_sqrt @ A @ D_inv_sqrt
    hat_A_coo    = hat_A.tocoo()
    indices      = torch.LongTensor(np.vstack([hat_A_coo.row, hat_A_coo.col]))
    values       = torch.FloatTensor(hat_A_coo.data)
    return torch.sparse_coo_tensor(indices, values, (N, N))


class WBPRLightGCN(nn.Module):
    def __init__(self, n_users, n_items, embed_dim, n_layers, hat_A):
        super().__init__()
        self.n_users  = n_users
        self.n_items  = n_items
        self.n_layers = n_layers
        self.user_emb = nn.Embedding(n_users, embed_dim)
        self.item_emb = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)
        self.hat_A = hat_A   # 固定在 CPU

    def forward(self):
        dev = self.user_emb.weight.device
        E0  = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        all_layers = [E0]
        E = E0
        for _ in range(self.n_layers):
            E = torch.sparse.mm(self.hat_A, E.cpu()).to(dev)
            all_layers.append(E)
        E_final    = torch.stack(all_layers, dim=1).mean(dim=1)
        return E_final[:self.n_users], E_final[self.n_users:]

    def bpr_loss(self, user_final, item_final, uids, pos_iids, neg_iids):
        u_emb   = user_final[uids]
        pos_emb = item_final[pos_iids]
        neg_emb = item_final[neg_iids]
        loss    = -torch.log(torch.sigmoid(
            (u_emb * pos_emb).sum(1) - (u_emb * neg_emb).sum(1)
        ) + 1e-8).mean()
        reg = (self.user_emb.weight[uids].norm(2).pow(2) +
               self.item_emb.weight[pos_iids].norm(2).pow(2) +
               self.item_emb.weight[neg_iids].norm(2).pow(2)) / len(uids)
        return loss + WEIGHT_DECAY * reg


def build_user_pos_dict(df):
    d = defaultdict(set)
    for row in df.itertuples(index=False):
        d[row.uid].add(row.iid)
    return d


def sample_bpr_batch(all_uids, all_iids, user_pos_dict, n_items, batch_size):
    idx      = np.random.choice(len(all_uids), size=batch_size, replace=True)
    uids     = all_uids[idx]
    pos_iids = all_iids[idx]
    neg_iids = np.random.randint(0, n_items, size=batch_size)
    for j in range(batch_size):
        while neg_iids[j] in user_pos_dict[uids[j]]:
            neg_iids[j] = np.random.randint(0, n_items)
    return uids, pos_iids, neg_iids


# ─────────────────────────────────────────────────────────────
# 5. Retrain
# ─────────────────────────────────────────────────────────────
print("\n[5] 开始 Retrain...")
hat_A     = build_normalized_adjacency(df_retrain, n_users, n_items, RATING_MAX)
model     = WBPRLightGCN(n_users, n_items, EMBED_DIM, N_LAYERS, hat_A).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

all_uids      = df_retrain["uid"].values
all_iids      = df_retrain["iid"].values
user_pos_dict = build_user_pos_dict(df_retrain)
n_batches     = max(1, len(all_uids) // BATCH_SIZE)

best_loss  = float("inf")
avg_loss   = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()
    for _ in range(n_batches):
        uids_b, pos_b, neg_b = sample_bpr_batch(
            all_uids, all_iids, user_pos_dict, n_items, BATCH_SIZE)
        uids_t = torch.LongTensor(uids_b).to(device)
        pos_t  = torch.LongTensor(pos_b).to(device)
        neg_t  = torch.LongTensor(neg_b).to(device)
        optimizer.zero_grad()
        uf, itf = model()
        loss = model.bpr_loss(uf, itf, uids_t, pos_t, neg_t)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / n_batches
    if epoch % LOG_EVERY == 0 or epoch == 1:
        print(f"  Epoch {epoch:>3d}/{EPOCHS}  loss={avg_loss:.4f}  [{time.time()-t0:.1f}s]")
    if avg_loss < best_loss:
        best_loss = avg_loss

# 提取最终 embedding
model.eval()
with torch.no_grad():
    user_final_retrain, item_final_retrain = model()
user_emb_retrain = user_final_retrain.cpu().numpy()
item_emb_retrain = item_final_retrain.cpu().numpy()

torch.save(
    {"user_embeddings": user_final_retrain.cpu(),
     "item_embeddings": item_final_retrain.cpu(),
     "loss": avg_loss},
    os.path.join(SAVE_DIR, "embeddings_retrain.pt")
)
print(f"\n  Retrain 完成，最低 loss={best_loss:.4f}")
print(f"  embedding 已保存至 {SAVE_DIR}/embeddings_retrain.pt")




<>:1: SyntaxWarning: invalid escape sequence '\ '
<>:1: SyntaxWarning: invalid escape sequence '\ '
/var/folders/rz/8jmyj8817yz96t1prby58jc40000gn/T/ipykernel_3113/1201620856.py:1: SyntaxWarning: invalid escape sequence '\ '
  """


[1] 加载数据...
  设备: mps
  n_users=27794, n_items=144853

[2] 读取原始训练数据...
  原始训练集大小: 363218

[3] 构建 S_u 与 D_remove...
  D_remove 大小: 3696 条交互

[4] 构建 D_retrain...
  D_retrain 大小: 359497 条（删除了 3721 条）

[5] 开始 Retrain...


/var/folders/rz/8jmyj8817yz96t1prby58jc40000gn/T/ipykernel_3113/1201620856.py:173: RuntimeWarning: divide by zero encountered in divide
  d_w_inv_sqrt = np.where(d_w > 0, 1.0 / np.sqrt(d_w), 0.0)


  Epoch   1/100  loss=0.6711  [69.8s]
  Epoch  10/100  loss=0.1684  [69.7s]
  Epoch  20/100  loss=0.0637  [71.1s]
  Epoch  30/100  loss=0.0324  [91.8s]
  Epoch  40/100  loss=0.0201  [69.1s]
  Epoch  50/100  loss=0.0145  [68.8s]
  Epoch  60/100  loss=0.0119  [68.0s]
  Epoch  70/100  loss=0.0103  [68.7s]
  Epoch  80/100  loss=0.0093  [68.2s]
  Epoch  90/100  loss=0.0087  [68.0s]
  Epoch 100/100  loss=0.0083  [68.0s]

  Retrain 完成，最低 loss=0.0082
  embedding 已保存至 /Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/Retrain/embeddings_retrain.pt


In [8]:
# ─────────────────────────────────────────────────────────────
# 6. 评估指标
# ─────────────────────────────────────────────────────────────
print("\n[6] 计算评估指标...")

BASIS_PATH = "/Users/yubinhao/ml-1m/Amazon Review/LightGCN加权BPR/pattern_basis/pattern_basis.pt"
basis_ckpt = torch.load(BASIS_PATH, map_location="cpu", weights_only=False)

MIN_NT = 0.01

# 构建 I_target / I_nontarget
I_target_dict    = {}
I_nontarget_dict = {}
for uid, patterns in S_u_dict.items():
    B_u     = user_item_sets.get(uid, set())
    I_union = set()
    for k in patterns:
        I_union |= set(pattern_item_sets[k])
    I_target_dict[uid]    = B_u & I_union
    I_nontarget_dict[uid] = B_u - I_target_dict[uid]

valid_uids = [uid for uid in sampled_uids
              if len(S_u_dict[uid]) > 0 and len(I_target_dict[uid]) > 0]
print(f"  有效用户数: {len(valid_uids)}")

results = []
for uid in valid_uids:
    eu_orig    = user_emb_orig[uid]
    eu_unlearn = user_emb_unlearn[uid]
    eu_retrain = user_emb_retrain[uid]

    I_target    = list(I_target_dict[uid])
    I_nontarget = list(I_nontarget_dict[uid])

    # Target Score Drop
    s_orig_t    = item_emb_orig[I_target]    @ eu_orig
    s_unlearn_t = item_emb_orig[I_target]    @ eu_unlearn
    s_retrain_t = item_emb_retrain[I_target] @ eu_retrain
    drop_unlearn = float((s_orig_t - s_unlearn_t).mean())
    drop_retrain = float((s_orig_t - s_retrain_t).mean())

    # Non-target Score Change
    if len(I_nontarget) > 0:
        s_orig_nt    = item_emb_orig[I_nontarget]    @ eu_orig
        s_unlearn_nt = item_emb_orig[I_nontarget]    @ eu_unlearn
        s_retrain_nt = item_emb_retrain[I_nontarget] @ eu_retrain
        nt_change_unlearn = float(np.abs(s_orig_nt - s_unlearn_nt).mean())
        nt_change_retrain = float(np.abs(s_orig_nt - s_retrain_nt).mean())
    else:
        nt_change_unlearn = nt_change_retrain = 0.0

    # Selectivity Ratio（分母过小时置 nan）
    sel_unlearn = drop_unlearn / nt_change_unlearn if nt_change_unlearn > MIN_NT else np.nan
    sel_retrain = drop_retrain / nt_change_retrain if nt_change_retrain > MIN_NT else np.nan

    # Target Score Gap
    gap = abs(drop_unlearn - drop_retrain)

    # PFR_multi (Unlearn vs Retrain)
    pfr_unlearn_list = []
    pfr_retrain_list = []
    for k in S_u_dict[uid]:
        vk       = basis_ckpt["pattern_basis"].numpy()[k]
        a_before = np.dot(eu_orig, vk)
        if abs(a_before) <= 1e-10:
            pfr_unlearn_list.append(0.0)
            pfr_retrain_list.append(0.0)
            continue
        pfr_unlearn_list.append(1.0 - np.dot(eu_unlearn, vk) / a_before)
        pfr_retrain_list.append(1.0 - np.dot(eu_retrain, vk) / a_before)
    pfr_unlearn = float(np.mean(pfr_unlearn_list))
    pfr_retrain = float(np.mean(pfr_retrain_list))

    # Overlap@K (Unlearning vs Retrain)
    topk_unlearn = set(np.argpartition(item_emb_orig    @ eu_unlearn, -TOP_K)[-TOP_K:])
    topk_retrain = set(np.argpartition(item_emb_retrain @ eu_retrain, -TOP_K)[-TOP_K:])
    overlap_ur   = len(topk_unlearn & topk_retrain) / TOP_K

    results.append({
        "uid":                    uid,
        "n_target_patterns":      len(S_u_dict[uid]),
        "n_target_items":         len(I_target),
        "n_nontarget_items":      len(I_nontarget),
        "drop_unlearn":           drop_unlearn,
        "drop_retrain":           drop_retrain,
        "gap_target":             gap,
        "nt_change_unlearn":      nt_change_unlearn,
        "nt_change_retrain":      nt_change_retrain,
        "sel_unlearn":            sel_unlearn,
        "sel_retrain":            sel_retrain,
        "pfr_unlearn":            pfr_unlearn,
        "pfr_retrain":            pfr_retrain,
        "overlap_unlearn_retrain": overlap_ur,
    })

df_results = pd.DataFrame(results)

print(f"\n[汇总] {len(df_results)} 个有效用户")
print("-" * 65)
metrics = [
    ("Target Score Drop   (Unlearn)",       "drop_unlearn"),
    ("Target Score Drop   (Retrain)",       "drop_retrain"),
    ("Target Score Gap",                    "gap_target"),
    ("Non-target Change   (Unlearn)",       "nt_change_unlearn"),
    ("Non-target Change   (Retrain)",       "nt_change_retrain"),
    ("Selectivity Ratio   (Unlearn)",       "sel_unlearn"),
    ("Selectivity Ratio   (Retrain)",       "sel_retrain"),
    ("PFR_multi           (Unlearn)",       "pfr_unlearn"),
    ("PFR_multi           (Retrain)",       "pfr_retrain"),
    (f"Overlap@{TOP_K} Unlearn vs Retrain", "overlap_unlearn_retrain"),
]
for label, col in metrics:
    v = df_results[col].dropna()
    print(f"  {label:<38} mean={v.mean():.4f}, std={v.std():.4f}, "
          f"min={v.min():.4f}, max={v.max():.4f}  (n={len(v)})")


[6] 计算评估指标...
  有效用户数: 889

[汇总] 889 个有效用户
-----------------------------------------------------------------
  Target Score Drop   (Unlearn)          mean=2.3735, std=1.2041, min=-1.9345, max=7.6047  (n=889)
  Target Score Drop   (Retrain)          mean=7.7701, std=2.9403, min=0.4927, max=20.8272  (n=889)
  Target Score Gap                       mean=5.4625, std=3.1021, min=0.0032, max=19.2065  (n=889)
  Non-target Change   (Unlearn)          mean=0.8682, std=0.5003, min=0.0000, max=2.8605  (n=889)
  Non-target Change   (Retrain)          mean=0.7327, std=0.4717, min=0.0000, max=3.6757  (n=889)
  Selectivity Ratio   (Unlearn)          mean=3.1997, std=2.1243, min=-8.3886, max=23.1903  (n=887)
  Selectivity Ratio   (Retrain)          mean=15.5504, std=13.5314, min=0.8953, max=141.3947  (n=887)
  PFR_multi           (Unlearn)          mean=0.9746, std=0.0282, min=0.9000, max=1.0000  (n=889)
  PFR_multi           (Retrain)          mean=0.8289, std=0.5721, min=-2.4129, max=4.6553  (n=889